# Async Generators — Quick Tutorial for FastAPI

**Layer targeted:** 1 (what) + 2 (how), with a short Layer 3 (internals) at the end.
Prerequisites you already own: generators (`yield`), iterators (`__iter__`/`__next__`), asyncio basics.

> Jupyter allows `await` / `async for` at the top level of a cell. In a plain `.py` file, wrap it in `asyncio.run(main())`.

## 1. The problem it solves

A normal generator hands you values one at a time, but **producing each value must be instant** — if it needs a network call or DB query, it blocks the whole event loop.

An **async generator** can *pause to wait* (`await`) **and** *pause to hand you a value* (`yield`). While it waits, the event loop serves other requests.

**Analogy:** a conveyor-belt sushi bar vs. a waiter. A normal generator is a belt that must have the next plate ready the instant you reach. An async generator is a waiter: brings one plate, goes to the kitchen for the next (you and other tables aren't frozen meanwhile), returns when it's ready.

**Rule of thumb:** `async def` + `yield` = async generator. Consume with `async for`.

| | Sync | Async |
|---|---|---|
| Protocol | `__iter__` → `__next__` | `__aiter__` → `__anext__` (awaitable) |
| Ends with | `StopIteration` | `StopAsyncIteration` |
| Loop | `for` | `async for` |
| Generator form | `def` + `yield` | `async def` + `yield` |

## 2. Minimal example

In [1]:
import asyncio
import logging
from collections.abc import AsyncGenerator, AsyncIterator

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logger = logging.getLogger(__name__)


async def countdown(start: int, delay: float = 0.1) -> AsyncIterator[int]:
    """Yield start..1, awaiting between values (simulates waiting on I/O)."""
    for number in range(start, 0, -1):
        await asyncio.sleep(delay)   # event loop is FREE during this wait
        yield number                 # hand a value to the caller


async for number in countdown(3):
    logger.info("got %d", number)

got 3
got 2
got 1


Annotate async generator functions as `AsyncIterator[T]` (or `AsyncGenerator[YieldT, SendT]`). Calling one **does not run any body code** and you **don't `await` the call**:

In [2]:
agen = countdown(3)
print(type(agen))                   # async_generator
print(hasattr(agen, "__anext__"))   # True

<class 'async_generator'>
True


In [3]:
try:
    await agen                      # common mistake
except TypeError as exc:
    print("TypeError:", exc)

TypeError: object async_generator can't be used in 'await' expression


## 3. The protocol by hand (what `async for` really does)

```python
it = obj.__aiter__()
while True:
    try:
        x = await it.__anext__()
    except StopAsyncIteration:
        break
    ...  # loop body
```
An async generator object implements both methods for you (`__aiter__` returns itself, so it is **one-shot**, like a sync generator).

In [ ]:
class Ticker:
    """Class-based async iterator: what the generator gives you for free."""

    def __init__(self, limit: int, delay: float = 0.05) -> None:
        self._limit = limit
        self._delay = delay
        self._current = 0

    def __aiter__(self) -> "Ticker":
        return self

    async def __anext__(self) -> int:
        if self._current >= self._limit:
            raise StopAsyncIteration
        await asyncio.sleep(self._delay)
        self._current += 1
        return self._current


print([tick async for tick in Ticker(3)])

## 4. Async comprehensions and `anext()`

In [ ]:
async def numbers(limit: int) -> AsyncIterator[int]:
    for n in range(limit):
        await asyncio.sleep(0)
        yield n


squares = [n * n async for n in numbers(5)]
evens = {n async for n in numbers(10) if n % 2 == 0}
print(squares, evens)

# anext() builtin (3.10+): pull one value; the default avoids StopAsyncIteration
agen = numbers(2)
print(await anext(agen), await anext(agen), await anext(agen, "done"))

## 5. Real-world: a paginated API as a lazy stream

Instead of loading every page into a list, expose a stream. The caller decides when to stop — pages never needed are never fetched.
(Simulated so the notebook runs offline; in real code swap in `httpx.AsyncClient` / `AsyncSession`.)

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Page:
    items: list[dict]
    next_cursor: int | None


async def fake_fetch_page(cursor: int, page_size: int = 3, total: int = 8) -> Page:
    """Stand-in for `await client.get(url, params={"cursor": cursor})`."""
    await asyncio.sleep(0.05)
    items = [{"id": i} for i in range(cursor, min(cursor + page_size, total))]
    next_cursor = cursor + page_size if cursor + page_size < total else None
    logger.info("  [fetched page at cursor=%d]", cursor)
    return Page(items, next_cursor)


async def iter_all_items() -> AsyncIterator[dict]:
    """Flatten pages into a stream of items, fetching lazily."""
    cursor: int | None = 0
    while cursor is not None:
        page = await fake_fetch_page(cursor)
        for item in page.items:
            yield item
        cursor = page.next_cursor


async for item in iter_all_items():
    logger.info("item %s", item)
    if item["id"] == 3:   # early stop -> the last page is never fetched
        break

The same idea with SQLAlchemy 2.0 async — the **DB-streaming pattern** (don't `fetchall()` a huge table):
```python
async def stream_orders(session: AsyncSession, batch_size: int = 500) -> AsyncIterator[Order]:
    result = await session.stream_scalars(
        select(Order).execution_options(yield_per=batch_size)
    )
    async for order in result:
        yield order
```

## 6. Cleanup — the part that bites people

`try/finally` inside an async generator works, but **when** `finally` runs depends on the generator being closed. `break` out of `async for` does **not** close it immediately — it's closed later (GC / event-loop asyncgen hooks), which is non-deterministic. Use `contextlib.aclosing` for guaranteed cleanup.

In [ ]:
from contextlib import aclosing


async def resource_stream() -> AsyncIterator[int]:
    logger.info("open connection")
    try:
        for n in range(100):
            await asyncio.sleep(0)
            yield n
    finally:
        # runs on normal end, exception, aclose(), or cancellation
        logger.info("close connection")


async with aclosing(resource_stream()) as stream:
    async for n in stream:
        if n == 2:
            break
logger.info("after loop")   # "close connection" is logged BEFORE this line

## 7. Two-way: `asend`, `athrow`, `aclose`
Rarely needed in web code, but a classic interview extension of generators.

In [ ]:
async def accumulator() -> AsyncGenerator[int, int]:
    total = 0
    try:
        while True:
            increment = yield total       # receives the value passed to asend()
            total += increment
    except GeneratorExit:
        logger.info("accumulator closed with total=%d", total)
        raise


acc = accumulator()
await anext(acc)            # prime: run to the first yield (like next() on sync gens)
print(await acc.asend(5))   # 5
print(await acc.asend(10))  # 15
await acc.aclose()          # raises GeneratorExit inside -> cleanup runs

## 8. FastAPI usage (reference code — needs `pip install fastapi uvicorn`; not executed here)

### 8.1 Streaming response (big export / LLM tokens / progress)
```python
from collections.abc import AsyncIterator
from fastapi import APIRouter, Depends
from fastapi.responses import StreamingResponse

router = APIRouter()

async def csv_rows(session: AsyncSession) -> AsyncIterator[str]:
    yield "id,total\n"
    result = await session.stream_scalars(select(Order).execution_options(yield_per=500))
    async for order in result:
        yield f"{order.id},{order.total}\n"

@router.get("/orders/export")
async def export_orders(session: AsyncSession = Depends(get_session)) -> StreamingResponse:
    return StreamingResponse(csv_rows(session), media_type="text/csv")
```

### 8.2 Server-Sent Events (live updates)
```python
async def event_stream(request: Request) -> AsyncIterator[str]:
    while not await request.is_disconnected():       # stop when the client leaves
        message = await queue.get()                  # asyncio.Queue / Redis pub-sub
        yield f"data: {json.dumps(message)}\n\n"    # SSE wire format

@router.get("/events")
async def events(request: Request) -> StreamingResponse:
    return StreamingResponse(event_stream(request), media_type="text/event-stream")
```

### 8.3 Dependencies with `yield` — you already use async generators
```python
async def get_session() -> AsyncIterator[AsyncSession]:
    async with async_session_factory() as session:   # setup
        yield session                                # injected into the endpoint
    # teardown runs after the response
```
FastAPI wraps this with `asynccontextmanager` internally. The code after `yield` is teardown — the same shape as a `@pytest.fixture` with `yield`.

### 8.4 `@asynccontextmanager` — async generator → `async with`
```python
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncIterator[None]:
    await redis.connect()
    yield
    await redis.close()

app = FastAPI(lifespan=lifespan)
```
The generator must have exactly one `yield`.

## 9. Gotchas (the interview traps)

1. **`return value` is illegal** in an async generator (`SyntaxError`). Bare `return` is fine. (Sync generators can return a value via `StopIteration.value`.)
2. **Don't `await` the call, don't use plain `for`:** `for x in agen()` → `TypeError: 'async_generator' object is not iterable`.
3. **One-shot:** a second iteration yields nothing.
4. **Blocking code inside freezes the loop.** `time.sleep()`, `requests.get()`, sync ORM calls inside an async generator stall every other request. Use `await asyncio.sleep`, `httpx.AsyncClient`, async drivers, or `run_in_executor`.
5. **Client disconnect → `asyncio.CancelledError`** is thrown into the generator at its current `await`. Never swallow it; clean up in `finally` and let it propagate.
6. **Session lifetime with `StreamingResponse`:** the body streams *after* the endpoint returns, and dependency-teardown timing has changed across FastAPI versions. If the generator uses a `Depends` session, verify it's still open while streaming, or open the session inside the generator.
7. **Status and headers are sent before the first chunk.** An error mid-stream can't turn the 200 into a 500 — the connection just breaks. Run risky validation/queries *before* the first `yield`.
8. **Early `break` ≠ immediate cleanup** — use `aclosing()` (section 6).
9. **Cancel scopes across a `yield`** (`asyncio.timeout`, `TaskGroup` spanning a `yield`) are fragile — keep them entirely on one side of the `yield`.

In [ ]:
# Gotchas 1 and 2, demonstrated
try:
    compile("async def bad():\n    yield 1\n    return 2", "<demo>", "exec")
except SyntaxError as exc:
    print("SyntaxError:", exc.msg)

try:
    for _ in numbers(2):
        pass
except TypeError as exc:
    print("TypeError:", exc)

## 10. Under the hood (Layer 3, brief)

- Calling an async generator function returns an `async_generator` object; **no body code runs** until the first `__anext__()`.
- `__anext__()` returns an *awaitable*; awaiting it runs the body until the next `yield` (→ value) or the end (→ `StopAsyncIteration`).
- `asyncio.run()` calls `loop.shutdown_asyncgens()` at exit, closing unfinished generators — the safety net behind non-deterministic cleanup.
- It is one frame with **two** suspension targets: `await` suspends to the event loop; `yield` suspends to the consumer.

## 11. Interview angle

**What it tests:** whether you separate *awaiting* (give control to the loop) from *yielding* (give a value to the caller).

- **L1:** "`async def` with `yield`, consumed with `async for`."
- **L2:** "Each item may involve I/O without blocking the loop: streaming DB rows, paginated APIs, SSE. FastAPI `yield` dependencies are async generators."
- **L3:** "Cleanup isn't deterministic on early exit — use `aclosing`. Can't `return` a value. Cancellation arrives as `CancelledError` at an `await`."
- **Follow-up:** *"Why not return a list from a coroutine?"* → memory, time-to-first-byte, early termination.
- **Trap:** believing `async for` runs iterations concurrently. It doesn't — items are produced **sequentially**. For concurrency use `asyncio.gather` / `TaskGroup`.

## 12. Try it yourself
1. Write `async def merge(*streams)` that yields items from several async iterators as they arrive (hint: `asyncio.Queue` + one task per stream).
2. Add `max_items` to `iter_all_items`; use `aclosing` to prove `finally` runs on early exit.
3. Build a FastAPI SSE endpoint that stops on `request.is_disconnected()` and logs from `finally`.

**One-liner:** *an async generator is a generator that can `await` between `yield`s — use it to stream I/O-bound data lazily, and `aclosing` it when you may bail early.*